# 03_sensor_streaming_anomaly_detection

**Purpose:** Demo of a sensor anomaly rule. Simulates pressure readings from Silver PROD rows (random values,
about 5% forced negative) and flags wells with 3 consecutive negative readings. Not part of the Gold pipeline.

**Input tables:** `silver_petrinex_vol`.

**Output tables:** `dq_results`, `rejected_records` (append, rule `sensor_malfunction_consecutive_neg`).

**Run order:** optional, any time after 02_silver_transform. Independent of 03_gold_transform.

In [2]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.window import Window

RUN_ID = str(uuid.uuid4())[:8]

if spark.catalog.tableExists("silver_petrinex_vol"):
    sp = spark.sql("SELECT * FROM silver_petrinex_vol")
else:
    raise ValueError("Table silver_petrinex_vol does not exist. Please run silver transformation first.")

sensor_stream = (sp
    .filter(F.col("ActivityID") == "PROD")
    .withColumn("sensor_timestamp", F.current_timestamp())
    .withColumn("simulated_pressure", F.col("volume") * 1.5 + F.rand() * 10)
    .withColumn("simulated_pressure", 
                F.when(F.rand() < 0.05, -10.0).otherwise(F.col("simulated_pressure")))
)

time_window = Window.partitionBy("well_key").orderBy("sensor_timestamp").rowsBetween(-2, 0)

analyzed_stream = (sensor_stream
    .withColumn("is_negative", F.col("simulated_pressure") < 0)
    .withColumn("consecutive_negative_count", F.sum(F.col("is_negative").cast("int")).over(time_window))
    .withColumn("sensor_malfunctioning", F.col("consecutive_negative_count") >= 3)
)

anomalies = analyzed_stream.filter(F.col("sensor_malfunctioning") == True)

# explicitly cast to decimal(18,3) to match the data type of the original rejected_records table
defect_records = (anomalies
    .select(
        F.col("production_month"),
        F.col("ReportingFacilityID"),
        F.col("well_key"),
        F.col("simulated_pressure").cast("decimal(18,3)").alias("volume"),
        F.col("_source_file")
    )
    .withColumn("rule_name", F.lit("sensor_malfunction_consecutive_neg"))
    .withColumn("run_id", F.lit(RUN_ID))
)

dq_summary_row = [(
    RUN_ID,
    "gold",
    "sensor_stream",
    "sensor_malfunction_consecutive_neg",
    "reject",
    int(anomalies.count())
)]

dq_df = spark.createDataFrame(
    dq_summary_row, 
    "run_id string, layer string, table_name string, rule_name string, severity string, rows_flagged long"
)

dq_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("dq_results")
defect_records.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("rejected_records")
print("RUN_ID:", RUN_ID)
print("Total analyzed rows:", analyzed_stream.count())
print("Detected sensor anomalies:", anomalies.count())
display(spark.table("dq_results").filter(F.col("run_id") == RUN_ID))

StatementMeta(, 30344fca-b657-41ee-9bde-6be0caffe099, 4, Finished, Available, Finished, False)

RUN_ID: 640b3a97
Total analyzed rows: 484071
Detected sensor anomalies: 24


SynapseWidget(Synapse.DataFrame, b04dd15b-03d9-4bb9-9129-4a1d9a8695de)